# 笔记本 04 — 动量策略

**阶段 2 · 策略模块 (1 / 4)**

---

## 🎯 学习目标

| # | 目标 |
|---|------|
| 1 | 理解动量交易背后的经济学直觉 |
| 2 | 计算多回看期收益率得分 |
| 3 | 构建 RSI、EMA 和成交量**过滤器**以剔除低质量候选标的 |
| 4 | 用复合得分对资产排名，并进行最小-最大归一化 |
| 5 | 将我们的从零实现与生产环境的 `rank_assets_by_momentum()` 对比 |

### 前置要求
- NB01（市场数据获取）
- NB02（RSI、EMA、布林带）
- NB03（Sharpe / Sortino 基础，用于后续评估）

In [ ]:
# ── 初始化 ─────────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
print("✅ 导入完毕  |  项目根目录:", ROOT)

---
## 1 · 什么是动量？

**核心思想：** 在短到中期时间范围内，近期上涨的资产往往会继续上涨（反之亦然）。这是金融学中最有据可查的异象之一（Jegadeesh & Titman 1993）。

我们的机器人实现了一种**横截面动量**策略：

1. 在**多个回看窗口**（默认 3、5、7 天）上计算收益率得分。
2. 对多回看期收益率取均值 → **复合动量得分**。
3. 应用质量**过滤器**（RSI、EMA、成交量）以剔除噪声候选标的。
4. 对存活资产**排名**并选取 top-N。
5. 对得分进行**最小-最大归一化**到 [0, 1]，用于权重分配。

### 为什么用多回看期？

单一回看期很脆弱。对 3 天、5 天、7 天收益率取均值可以平滑噪声：

$$
\text{composite}(i) = \frac{1}{K} \sum_{k=1}^{K} r_{i,L_k}
\quad\text{其中 }\; r_{i,L_k} = \frac{P_{i,t}}{P_{i,t-L_k}} - 1
$$

其中 $K=3$，$L \in \{3,5,7\}$，来自 `config/strategy_params.yaml`。

---
## 2 · 生成合成价格数据

我们将创建一个逼真的多资产价格面板，使每个单元格无需 API 密钥即可运行。

In [ ]:
# ── 合成价格面板 ───────────────────────────────────────────
np.random.seed(42)
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "ADAUSDT",
           "XRPUSDT", "DOGEUSDT", "AVAXUSDT", "DOTUSDT",
           "LINKUSDT", "MATICUSDT", "NEARUSDT", "APTUSDT"]
DAYS = 60
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=DAYS, freq="D")

# 每个资产：带微小漂移的几何随机游走
base_prices = [60000, 3500, 140, 0.45, 0.55, 0.15,
               35, 7, 14, 0.9, 5.5, 9]
drifts  = [0.003, 0.005, 0.008, -0.001, 0.001, 0.002,
           0.006, -0.002, 0.004, -0.003, 0.007, 0.009]
vols    = [0.025, 0.035, 0.05, 0.04, 0.03, 0.06,
           0.045, 0.04, 0.038, 0.05, 0.055, 0.048]

closes = pd.DataFrame(index=dates)
volumes = pd.DataFrame(index=dates)
for sym, p0, mu, sigma in zip(SYMBOLS, base_prices, drifts, vols):
    log_returns = np.random.normal(mu, sigma, DAYS)
    closes[sym]  = p0 * np.exp(np.cumsum(log_returns))
    volumes[sym] = np.random.uniform(5e6, 80e6, DAYS)  # 以 USD 计的报价成交量

print(f"面板形状: {closes.shape}  ({len(SYMBOLS)} 资产 × {DAYS} 天)")
closes.tail(3)

---
## 3 · 逐步计算动量得分

### 3.1  单回看期收益率

In [ ]:
def lookback_return(prices: pd.Series, lookback: int) -> float:
    """计算最近 `lookback` 期的收益率。
    
    r = P_t / P_{t-L} - 1
    """
    if len(prices) < lookback + 1 or prices.iloc[-(lookback + 1)] == 0:
        return float("nan")
    return (prices.iloc[-1] / prices.iloc[-(lookback + 1)]) - 1.0

# 演示：BTC 的 3 日收益率
r3 = lookback_return(closes["BTCUSDT"], 3)
print(f"BTC 3日收益率: {r3:+.4f}  ({r3*100:+.2f}%)")

### 3.2  多回看期复合得分

对 `[3, 5, 7]` 天回看期的收益率取均值 — 与 `rank_assets_by_momentum()` 完全一致。

In [ ]:
LOOKBACKS = [3, 5, 7]  # 来自 strategy_params.yaml → momentum.lookback_days

def composite_momentum(prices: pd.Series, lookbacks: list[int]) -> float:
    """多回看期收益率的均值。"""
    returns = [lookback_return(prices, lb) for lb in lookbacks]
    valid = [r for r in returns if not np.isnan(r)]
    return np.mean(valid) if valid else float("nan")

# 为每个资产计算
raw_scores = {sym: composite_momentum(closes[sym], LOOKBACKS) for sym in SYMBOLS}
score_df = (
    pd.Series(raw_scores, name="composite_score")
    .to_frame()
    .sort_values("composite_score", ascending=False)
)
score_df["rank"] = range(1, len(score_df) + 1)
score_df.style.format({"composite_score": "{:+.4f}"})

---
## 4 · 质量过滤器

原始动量得分噪声较大。我们的生产代码在排名前应用了**三个过滤器**：

| 过滤器 | 条件 | 目的 |
|--------|------|------|
| **RSI** | `RSI > 45` | 剔除动量衰减的资产 |
| **EMA** | `价格 > EMA(20)` | 确认上升趋势 |
| **成交量** | `报价成交量 ≥ 1000万美元` | 确保流动性 |

一个资产必须**同时通过三项**才能存活。

### 4.1  为什么选择这些特定过滤器？

- **RSI > 45**（而非 50）：留一个小缓冲区，避免在中线附近反复震荡（whipsaw）。
- **价格 > EMA-20**：经典趋势跟踪确认 — 不买入在短期均线下方交易的资产。
- **成交量 ≥ 1000万美元**：薄弱市场买卖价差大、滑点高；机器人必须高效执行订单。

这些阈值定义在 `config/strategy_params.yaml` 中。

In [ ]:
# ── RSI 计算（来自 NB02 / 生产代码）─────────────────────
def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    delta = prices.astype(float).diff()
    gains = delta.clip(lower=0.0)
    losses = (-delta).clip(lower=0.0)
    avg_gain = gains.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = losses.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss_safe = avg_loss.mask(avg_loss == 0.0)
    rs = avg_gain / avg_loss_safe
    rsi = 100.0 - (100.0 / (1.0 + rs))
    rsi = rsi.mask((avg_loss == 0.0) & (avg_gain > 0.0), 100.0)
    rsi = rsi.mask((avg_gain == 0.0) & (avg_loss > 0.0), 0.0)
    rsi = rsi.mask((avg_gain == 0.0) & (avg_loss == 0.0), 50.0)
    return rsi.astype(float)

# ── 应用三个过滤器 ─────────────────────────────────────────
RSI_THRESHOLD  = 45.0           # 来自 strategy_params.yaml
EMA_PERIOD     = 20
MIN_VOLUME_USD = 10_000_000.0

filter_records = []
for sym in SYMBOLS:
    price_series  = closes[sym].dropna()
    vol_series    = volumes[sym].dropna()
    
    current_price   = float(price_series.iloc[-1])
    current_ema     = float(price_series.ewm(span=EMA_PERIOD, adjust=False).mean().iloc[-1])
    current_rsi     = float(calculate_rsi(price_series).iloc[-1])
    current_volume  = float(vol_series.iloc[-1])
    
    pass_rsi    = current_rsi > RSI_THRESHOLD
    pass_ema    = current_price > current_ema
    pass_volume = current_volume >= MIN_VOLUME_USD
    
    filter_records.append({
        "symbol":  sym,
        "price":   current_price,
        "EMA-20":  current_ema,
        "RSI-14":  current_rsi,
        "vol_USD":  current_volume,
        "pass_rsi": pass_rsi,
        "pass_ema": pass_ema,
        "pass_vol": pass_volume,
        "ALL_PASS": pass_rsi and pass_ema and pass_volume,
    })

filter_df = pd.DataFrame(filter_records).set_index("symbol")

# 颜色编码：绿色通过，红色未通过
def _colour_pass_fail(val):
    if isinstance(val, bool):
        return "color: green" if val else "color: red; font-weight: bold"
    return ""

filter_df.style.applymap(_colour_pass_fail).format({
    "price": "{:.4f}", "EMA-20": "{:.4f}",
    "RSI-14": "{:.1f}", "vol_USD": "{:,.0f}"
})

In [ ]:
# ── 存活资产 ───────────────────────────────────────────────
survivors = filter_df[filter_df["ALL_PASS"]].index.tolist()
print(f"通过质量过滤后存活: {len(survivors)} / {len(SYMBOLS)}")
print(survivors)

---
## 5 · 排名与最小-最大归一化

过滤之后，我们：
1. 按复合得分对存活资产排序。
2. 取 **top-N**（`strategy_params.yaml` 默认为 8）。
3. 归一化到 $[0, 1]$：

$$
\text{norm}_i = \frac{s_i - s_{\min}}{s_{\max} - s_{\min}}
$$

当所有得分相等（$s_{\max} = s_{\min}$）时，每个资产获得 `normalized_score = 1.0`。

In [ ]:
TOP_N = 8  # 来自 strategy_params.yaml → momentum.top_n_assets

# 仅存活资产的复合得分
survivor_scores = {
    sym: raw_scores[sym]
    for sym in survivors
    if not np.isnan(raw_scores.get(sym, float("nan")))
}
sorted_survivors = sorted(survivor_scores.items(),
                          key=lambda x: x[1], reverse=True)[:TOP_N]

# 最小-最大归一化
scores_only = [s for _, s in sorted_survivors]
s_min, s_max = min(scores_only), max(scores_only)
span = s_max - s_min

ranking = []
for rank, (sym, score) in enumerate(sorted_survivors, 1):
    norm = 1.0 if span == 0 else (score - s_min) / span
    ranking.append({
        "rank": rank,
        "symbol": sym,
        "composite_score": score,
        "normalized_score": norm,
    })

ranking_df = pd.DataFrame(ranking).set_index("rank")
ranking_df.style.format({"composite_score": "{:+.5f}", "normalized_score": "{:.4f}"})

---
## 6 · 动量可视化

### 6.1  柱状图：复合得分

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- 左侧：所有资产的复合得分 ----
ax = axes[0]
all_scores = pd.Series(raw_scores).sort_values(ascending=True)
colours = ["#2ecc71" if s > 0 else "#e74c3c" for s in all_scores]
all_scores.plot.barh(ax=ax, color=colours)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("复合动量得分")
ax.set_title("所有资产 – 原始复合得分")

# --- 右侧：Top-N 的归一化得分 ----
ax2 = axes[1]
norm_series = ranking_df.set_index("symbol")["normalized_score"].sort_values()
norm_series.plot.barh(ax=ax2, color="#3498db")
ax2.set_xlabel("归一化得分 [0, 1]")
ax2.set_title(f"Top-{TOP_N} 存活资产 – 归一化得分")

plt.tight_layout()
plt.show()

### 6.2  Top-N 价格走势

In [ ]:
top_symbols = ranking_df["symbol"].tolist()

# 将所有价格序列归一化为起始值 100 以便比较
norm_prices = closes[top_symbols].apply(lambda s: s / s.iloc[0] * 100)

fig, ax = plt.subplots(figsize=(14, 6))
for sym in top_symbols:
    ax.plot(norm_prices.index, norm_prices[sym], label=sym, lw=1.5)
ax.axhline(100, color="gray", ls="--", lw=0.8, label="基准 = 100")
ax.set_ylabel("归一化价格（起始 = 100）")
ax.set_title("顶部动量资产的价格走势")
ax.legend(ncol=4, fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()
plt.show()

---
## 7 · 敏感性分析

回看窗口的选择如何影响排名？让我们扫描不同配置。

In [ ]:
lookback_configs = {
    "短期 [2,3,4]": [2, 3, 4],
    "默认 [3,5,7]": [3, 5, 7],
    "中期 [5,10,15]": [5, 10, 15],
    "长期 [7,14,21]": [7, 14, 21],
}

sensitivity = {}
for label, lbs in lookback_configs.items():
    scores = {
        sym: composite_momentum(closes[sym], lbs)
        for sym in survivors
    }
    ranked = sorted(scores, key=scores.get, reverse=True)
    sensitivity[label] = {sym: rank+1 for rank, sym in enumerate(ranked)}

sens_df = pd.DataFrame(sensitivity)
sens_df.index.name = "Symbol"
print("按回看配置的排名（数字越小越好）：")
sens_df

**观察：** 具有持续动量的资产在*所有*回看窗口下都排名靠前 — 这些是最可靠的候选标的。

---
## 8 · 与生产代码对比

让我们调用 `bot/signals/momentum.py` 中的**实际生产函数** `rank_assets_by_momentum()` 并进行对比。

In [ ]:
from bot.signals.momentum import rank_assets_by_momentum, MomentumSignal

# 使用相同数据调用生产函数
production_signals: list[MomentumSignal] = rank_assets_by_momentum(
    closes,
    volumes,
    lookback_periods=(3, 5, 7),
    rsi_period=14,
    rsi_threshold=45.0,
    ema_period=20,
    min_volume_usd=10_000_000.0,
    top_n_assets=8,
)

print(f"生产函数返回 {len(production_signals)} 个信号\n")
prod_df = pd.DataFrame([
    {"rank": i+1, "symbol": s.symbol,
     "composite": s.composite_score,
     "normalised": s.normalized_score,
     "RSI": s.rsi, "EMA": s.ema, "volume": s.quote_volume}
    for i, s in enumerate(production_signals)
]).set_index("rank")

prod_df.style.format({
    "composite": "{:+.5f}", "normalised": "{:.4f}",
    "RSI": "{:.1f}", "EMA": "{:.4f}", "volume": "{:,.0f}"
})

In [ ]:
# ── 验证我们的手写排名与生产函数匹配 ─────────────────────
our_ranking  = ranking_df["symbol"].tolist()
prod_ranking = [s.symbol for s in production_signals]

# 它们应该在存活资产上达成一致（相同过滤器，相同公式）
shared = set(our_ranking) & set(prod_ranking)
print(f"手写排名:    {our_ranking}")
print(f"生产排名:    {prod_ranking}")
print(f"重叠:        {len(shared)} / {max(len(our_ranking), len(prod_ranking))}")
print()

# 比较复合得分
for s in production_signals:
    if s.symbol in survivor_scores:
        ours = survivor_scores[s.symbol]
        diff = abs(ours - s.composite_score)
        print(f"  {s.symbol:10s}  手写={ours:+.6f}  生产={s.composite_score:+.6f}  Δ={diff:.2e}")

---
## 9 · `MomentumSignal` 解剖

生产代码返回一个**冻结数据类**：

```python
@dataclass(frozen=True, slots=True)
class MomentumSignal:
    symbol: str
    composite_score: float   # 多回看期收益率均值
    normalized_score: float  # 最小-最大归一化到 [0, 1]
    price: float             # 最新收盘价
    ema: float               # 最新 K 线的 EMA-20
    rsi: float               # 最新 K 线的 RSI-14
    quote_volume: float      # 最新报价成交量（USD）
```

**设计要点：**
- `frozen=True` 防止意外修改 — 信号是**不可变事实**。
- `slots=True` 节省内存（在轮询数百个资产时很重要）。
- 信号自身携带过滤值（`rsi`、`ema`、`quote_volume`），下游模块可以检查资产*为什么*被选中。

---
## 10 · 滚动动量：时间序列视角

让我们计算每个资产随时间变化的滚动复合得分，观察排名如何变化。

In [ ]:
def rolling_composite(prices: pd.Series, lookbacks: list[int]) -> pd.Series:
    """在每个 K 线处计算复合动量得分。"""
    min_lb = max(lookbacks) + 1
    scores = pd.Series(index=prices.index, dtype=float)
    for i in range(min_lb, len(prices)):
        window = prices.iloc[:i+1]
        scores.iloc[i] = composite_momentum(window, lookbacks)
    return scores

# 为前4个资产计算
top4 = ranking_df["symbol"].tolist()[:4]
fig, ax = plt.subplots(figsize=(14, 5))
for sym in top4:
    rolling = rolling_composite(closes[sym], LOOKBACKS)
    ax.plot(rolling.dropna(), label=sym, lw=1.5)
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_ylabel("复合动量得分")
ax.set_title("滚动动量得分随时间变化")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()
plt.show()

---
## 11 · 简易动量回测

让我们构建一个最简回测：每天做多 top-3 动量资产（等权重），衡量累计收益。

In [ ]:
# ── 简单日频动量回测 ───────────────────────────────────────
MIN_HISTORY = max(LOOKBACKS) + 1
TOP_K = 3

daily_returns = closes.pct_change()
strategy_returns = []

for i in range(MIN_HISTORY, len(closes)):
    # 在第 i 天收盘时，使用截至第 i 天的数据进行排名
    window = closes.iloc[:i+1]
    scores = {}
    for sym in SYMBOLS:
        s = composite_momentum(window[sym], LOOKBACKS)
        if not np.isnan(s):
            # 简单过滤：仅正动量
            if s > 0:
                scores[sym] = s
    
    # 按动量取 Top-K
    selected = sorted(scores, key=scores.get, reverse=True)[:TOP_K]
    
    if selected and i + 1 < len(closes):
        # 次日收益率（等权重）
        next_ret = daily_returns.iloc[i + 1][selected].mean()
        strategy_returns.append({
            "date": closes.index[i + 1],
            "return": next_ret,
            "holdings": selected,
        })

strat_df = pd.DataFrame(strategy_returns).set_index("date")
strat_df["cumulative"] = (1 + strat_df["return"]).cumprod()

# 基准：等权重持有所有 12 个资产
benchmark_ret = daily_returns.iloc[MIN_HISTORY+1:].mean(axis=1)
benchmark_cum = (1 + benchmark_ret).cumprod()

# 绘图
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(strat_df.index, strat_df["cumulative"], label=f"动量 Top-{TOP_K}", lw=2)
ax.plot(benchmark_cum.index, benchmark_cum, label="等权重基准", lw=2, ls="--")
ax.set_ylabel("累计收益")
ax.set_title("动量策略 vs 等权重基准")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()
plt.show()

# 统计数据
total_ret  = strat_df["cumulative"].iloc[-1] - 1
ann_vol    = strat_df["return"].std() * np.sqrt(365)
sharpe     = (strat_df["return"].mean() / strat_df["return"].std()) * np.sqrt(365) if strat_df["return"].std() > 0 else 0
print(f"\n动量策略: 总收益 = {total_ret:+.2%}, 年化波动率 = {ann_vol:.2%}, Sharpe ≈ {sharpe:.2f}")
bench_total = benchmark_cum.iloc[-1] - 1
print(f"基准:     总收益 = {bench_total:+.2%}")

---
## 12 · 关键要点

| 概念 | 详情 |
|------|------|
| **多回看期** | 对 3/5/7 天收益率取均值以减少噪声 |
| **质量过滤器** | RSI > 45、价格 > EMA-20、成交量 ≥ 1000万美元 |
| **最小-最大归一化** | 将得分映射到 [0,1]，用于下游权重分配 |
| **MomentumSignal** | 冻结数据类，携带得分 + 所有过滤值 |
| **配置驱动** | 所有阈值来自 `strategy_params.yaml` |

### 动量管线一览

```
全域 (N 个资产)
  │
  ├── 为每个资产计算 composite_score
  │     └── 回看期收益率均值 [3, 5, 7]
  │
  ├── 过滤: RSI > 45
  ├── 过滤: 价格 > EMA(20)
  ├── 过滤: 成交量 ≥ 1000万美元
  │
  ├── 按 composite_score 降序排列
  ├── 取 top-N
  └── 最小-最大归一化 → MomentumSignal[]
```

---
## 🔬 练习

1. **RSI 阈值扫描：** 将 `RSI_THRESHOLD` 从 30 到 60 每隔 5 变化一次，绘制每个级别有多少资产存活。哪个阈值能最大化简单回测的 Sharpe 比率？

2. **回看期选择：** 尝试 `[1, 3, 5]`（更灵敏）和 `[7, 14, 21]`（更缓慢）。每日排名变化次数（换手率）有何不同？

3. **动量崩溃：** 动量策略容易受到**突然反转**的影响（参阅
文献）。修改回测，添加止损：如果持仓在单日内下跌超过 3%，立即平仓。这是否改善了 Sharpe 比率？

4. **成交量过滤消融实验：** 完全移除成交量过滤器。回测表现是否变化？用合成数据 vs 真实数据有何不同？

---
## ✅ 知识检查

1. 为什么机器人对多个回看期的收益率取均值，而不使用单一回看期？
2. 当所有存活资产的复合得分相同时，`normalized_score` 会怎样？
3. 为什么 `MomentumSignal` 是**冻结**数据类？
4. 列举三个质量过滤器并解释各自的目的。
5. 你如何修改代码以偏好动量*正在改善*（正二阶导数）的资产？

---
## 🔗 下一步

**[NB05 — 均值回归与配对交易 →](05_均值回归与配对交易.ipynb)**

我们将探索*相反*的哲学 — 买入跌幅过大且可能回归均值的资产 — 并学习如何使用协整检验进行**配对交易**。